# AtmosAlert - Inteligência Espacial para Saúde Urbana

## Turma:1TSCOR 2025/2 
## Grupo: Data Insight
## Autores: 

|            Nome           |    RM    |
| ------------------------- | -------- |
| Daniela Pereira da Silva  | RM567079 |
| Fabio Ladislau do Amaral  | RM567292 |
| Regina Ferreira Bolsaneli | RM567828 |
| Thiago de Souza Perez     | RM568322 |
| Wagner Martins de Santana | RM567734 |

## Cenário
## Breve explicação sobre como queimadas geram fumaça que viaja até os centros urbanos, lotando hospitais. descrição do problema contextualizado com o uso de dados espaciais.

## Solução
### Como antecipar esse risco cruzando focos de calor espaciais com a direção dos ventos.

## ODSs Atendidas
### ODS 11 (Cidades Sustentáveis) e ODS 13 (Ação Contra a Mudança Global do Clima)

## Importação de Bibliotecas

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

## Origem e Dicionário de Dados
### Fonte 1: INPE / BDQueimadas (Focos de calor reais detectados por satélite).
### Fonte 2: NASA POWER (Velocidade e direção dos ventos a 50m).
### Fonte 3: Copernicus Sentinel-5P (Índice de Aerossóis/Fumaça).

Identificação da fonte de dados, carregamento e descrição das variáveis; 

## Ingestão de Dados

In [ ]:
# Passando o caminho dos arquivos CSV
caminho_queimadas = '../data/raw/bdqueimadas_agosto_2023.csv'
caminho_fumaca = '../data/raw/fumaca_copernicus_agosto_2023.csv'
caminho_vento = '../data/raw/vento_nasa_agosto_2023.csv'

# Carregando os arquivos CSV em DataFrames
df_queimadas = pd.read_csv(caminho_queimadas)
df_fumaca = pd.read_csv(caminho_fumaca)
df_vento = pd.read_csv(caminho_vento)

# Validação do carregamento (linhas x colunas)
print(f"Queimadas (INPE): {df_queimadas.shape[0]} linhas e {df_queimadas.shape[1]} colunas.")
print(f"Fumaça (Copernicus): {df_fumaca.shape[0]} linhas e {df_fumaca.shape[1]} colunas.")
print(f"Vento (NASA): {df_vento.shape[0]} linhas e {df_vento.shape[1]} colunas.\n")

Queimadas (INPE): 871 linhas e 10 colunas.
Fumaça (Copernicus): 180 linhas e 3 colunas.
Vento (NASA): 186 linhas e 4 colunas.



## Limpeza e Tratamento
### Texto explicando o rigor metodológico: tratamento de valores nulos (NaN), conversão de tipos de dados (datas em formato datetime), e padronização dos nomes das cidades e colunas.

In [32]:
# Verificação de valores ausentes e tipo dos dados
print(df_queimadas.info())
print(df_fumaca.info())

<class 'pandas.DataFrame'>
RangeIndex: 871 entries, 0 to 870
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   id_bdq     871 non-null    int64         
 1   foco_id    871 non-null    str           
 2   lat        871 non-null    float64       
 3   lon        871 non-null    float64       
 4   datahora   871 non-null    datetime64[us]
 5   pais       871 non-null    str           
 6   estado     871 non-null    str           
 7   Municipio  871 non-null    str           
 8   bioma      871 non-null    str           
 9   Data       871 non-null    datetime64[us]
dtypes: datetime64[us](2), float64(2), int64(1), str(5)
memory usage: 68.2 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Data           180 non-null    datetime64[us]
 1   Indice_F

In [ ]:
# Verificação de valores ausentes e tipo dos dados
print(df_vento.info())

<class 'pandas.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Velocidade_Vento_50m  186 non-null    float64       
 1   Direcao_Vento_50m     186 non-null    float64       
 2   Data                  186 non-null    datetime64[us]
 3   Municipio             186 non-null    str           
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 5.9 KB
None


In [25]:
# Conversão de dados de String para Data
df_queimadas['Data'] = pd.to_datetime(df_queimadas['Data'])
df_queimadas['datahora'] = pd.to_datetime(df_queimadas['datahora'])
df_fumaca['Data'] = pd.to_datetime(df_fumaca['Data'])
df_vento['Data'] = pd.to_datetime(df_vento['Data'])

# Padronizando o nome da coluna de município para bater com as outras bases
df_queimadas.rename(columns={'municipio': 'Municipio'}, inplace=True)

# Estatistica Descritiva

In [58]:
# Estatística Descritiva do dataframe de queimadas e concatenção no eixo das colunas (axis=1).
# Preenchimento de NaN onde não se aplica.
desc_queimadas = pd.concat([df_queimadas.describe(), df_queimadas.describe(include=['object'])], axis=1)

desc_queimadas.style\
    .format(precision=3, na_rep="—")\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

/tmp/ipykernel_7059/2459310105.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  desc_queimadas = pd.concat([df_queimadas.describe(), df_queimadas.describe(include=['object'])], axis=1)


,id_bdq,lat,lon,datahora,Data,foco_id,pais,estado,Municipio,bioma
count,871.000,871.000,871.000,871,871,871,871,871,871,871
mean,1645461650.290,-9.154,-63.668,2023-08-20 18:16:38.645235,2023-08-20 00:09:55.177956,—,—,—,—,—
min,1641478872.000,-12.307,-66.595,2023-08-01 18:13:00,2023-08-01 00:00:00,—,—,—,—,—
25%,1645207496.500,-9.739,-64.883,2023-08-19 17:41:00,2023-08-19 00:00:00,—,—,—,—,—
50%,1645856974.000,-9.370,-64.100,2023-08-22 18:09:00,2023-08-22 00:00:00,—,—,—,—,—
75%,1646322527.500,-8.535,-62.689,2023-08-24 17:54:00,2023-08-24 00:00:00,—,—,—,—,—
max,1647733049.000,-8.005,-44.795,2023-08-31 17:54:00,2023-08-31 00:00:00,—,—,—,—,—
std,1435540.512,0.693,2.282,—,—,—,—,—,—,—
unique,—,—,—,—,—,871,1,2,2,2
top,—,—,—,—,—,b8e14a4e-7664-37ee-b432-756a7c289d54,Brasil,RONDÔNIA,PORTO VELHO,Amazônia


In [59]:
# Estatística Descritiva do dataframe de fumaça e concatenção no eixo das colunas (axis=1).
# Preenchimento de NaN onde não se aplica.
desc_fumaca = pd.concat([df_fumaca.describe(), df_fumaca.describe(include=['object'])], axis=1)

desc_fumaca.style\
    .format(precision=3, na_rep="—")\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

/tmp/ipykernel_7059/4135019938.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  desc_fumaca = pd.concat([df_fumaca.describe(), df_fumaca.describe(include=['object'])], axis=1)


,Data,Indice_Fumaca,Municipio
count,180,180.000,180
mean,2023-08-15 00:16:00,-0.111,—
min,2023-08-01 00:00:00,-1.786,—
25%,2023-08-08 00:00:00,-0.489,—
50%,2023-08-15 00:00:00,-0.011,—
75%,2023-08-23 00:00:00,0.299,—
max,2023-08-29 00:00:00,1.817,—
std,—,0.607,—
unique,—,—,6
top,—,—,CAMPINAS


In [60]:
# Estatística Descritiva do dataframe de vento e concatenção no eixo das colunas (axis=1).
# Preenchimento de NaN onde não se aplica.
desc_vento = pd.concat([df_vento.describe(), df_vento.describe(include=['object'])], axis=1)

desc_vento.style\
    .format(precision=3, na_rep="—")\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

/tmp/ipykernel_7059/560714571.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  desc_vento = pd.concat([df_vento.describe(), df_vento.describe(include=['object'])], axis=1)


,Velocidade_Vento_50m,Direcao_Vento_50m,Data,Municipio
count,186.000,186.000,186,186
mean,4.701,154.981,2023-08-16 00:00:00,—
min,0.940,0.100,2023-08-01 00:00:00,—
25%,3.650,63.575,2023-08-08 00:00:00,—
50%,4.640,118.900,2023-08-16 00:00:00,—
75%,6.010,280.300,2023-08-24 00:00:00,—
max,8.940,359.500,2023-08-31 00:00:00,—
std,1.849,113.743,—,—
unique,—,—,—,6
top,—,—,—,PORTO VELHO


## Visualizações Iniciais

In [ ]:
## Lorem Ipsum

## Teste de Hipótese

In [ ]:
## Lorem Ipsum

## Correlação
### Texto analítico com conclusões, limitações e contribuição para o Global Solution. 

In [ ]:
## Lorem Ipsum

## Conclusões e Impacto para Tomada de Decisão

In [1]:
## Lorem Ipsum

## Link do protótipo web
·	Inserir aqui o link do protótipo web.